# Task 2 — Parameter Experiments and Feature Selection

Experiment with different feature thresholds, clustering algorithms, and machine learning models to optimize integrative genomics analysis using **real TCGA-BRCA data**.

**Experiments:**
- Feature selection thresholds (top 200/500/800 genes, top 50/100/150 proteins)
- Clustering algorithms (K-means, Hierarchical, DBSCAN)
- ML models (Random Forest, SVM, Gradient Boosting, MLP)

**Data**: 277 breast cancer samples from TCGA with matched expression, proteomics, and mutation data.

## 1. Initialize Project Environment

In [1]:
"""Setup and imports for parameter experiments."""
import logging
import sys
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
numpy 2.1.3
pandas 2.2.3


## 2. Define Configuration Parameters

In [2]:
@dataclass
class Task2Config:
    """Configuration for parameter experiments."""

    handle: str = "AndreiCod"
    export_dir: Path = Path("./artifacts")
    data_dir: Path = Path("./artifacts")  # Load from Task 1 outputs
    random_seed: int = 42

    # Feature selection thresholds to test (adjusted for real data: 1000 genes, 208 proteins)
    gene_thresholds: List[int] = None
    protein_thresholds: List[int] = None

    # PCA components
    n_pca_components: int = 10

    def __post_init__(self):
        self.export_dir.mkdir(parents=True, exist_ok=True)
        if self.gene_thresholds is None:
            self.gene_thresholds = [100, 200, 500, 800]  # Max 1000 available
        if self.protein_thresholds is None:
            self.protein_thresholds = [50, 100, 150, 200]  # Max 208 available

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        info["data_dir"] = str(info["data_dir"])
        return info


CONFIG = Task2Config()
CONFIG.describe()

{'handle': 'AndreiCod',
 'export_dir': 'artifacts',
 'data_dir': 'artifacts',
 'random_seed': 42,
 'gene_thresholds': [100, 200, 500, 800],
 'protein_thresholds': [50, 100, 150, 200],
 'n_pca_components': 10}

## 3. Load Preprocessed TCGA-BRCA Data from Task 1

In [3]:
# Load TCGA-BRCA data from Task 1
snp_df = pd.read_csv(CONFIG.data_dir / "task1_snp_data.csv", index_col=0)
expr_df = pd.read_csv(CONFIG.data_dir / "task1_expression_data.csv", index_col=0)
prot_df = pd.read_csv(CONFIG.data_dir / "task1_proteomics_data.csv", index_col=0)
pheno_df = pd.read_csv(CONFIG.data_dir / "task1_phenotypes.csv", index_col=0)

# Prepare target variable
y = pheno_df["phenotype"].map({"responder": 1, "non_responder": 0})

print(f"{'=' * 60}")
print("LOADED TCGA-BRCA DATA (from Task 1)")
print(f"{'=' * 60}")
print(f"SNP/Mutations: {snp_df.shape}")
print(f"Expression: {expr_df.shape}")
print(f"Proteomics: {prot_df.shape}")
print(f"Phenotypes: {pheno_df.shape}")
print(f"\nClass distribution:")
print(f"  Responders (Basal-like): {(y == 1).sum()}")
print(f"  Non-responders (Other): {(y == 0).sum()}")

LOADED TCGA-BRCA DATA (from Task 1)
SNP/Mutations: (277, 100)
Expression: (1000, 277)
Proteomics: (208, 277)
Phenotypes: (277, 8)

Class distribution:
  Responders (Basal-like): 55
  Non-responders (Other): 222


## 4. Feature Selection Functions

In [4]:
def select_top_variable_features(
    df: pd.DataFrame, n_top: int, feature_type: str = "features"
) -> pd.DataFrame:
    """Select top N most variable features based on variance."""
    variances = df.var(axis=1)
    top_features = variances.sort_values(ascending=False).head(n_top).index
    selected = df.loc[top_features]
    logging.info(f"Selected top {n_top} variable {feature_type}")
    return selected


def integrate_features(
    expr: pd.DataFrame, prot: pd.DataFrame, snp: pd.DataFrame
) -> pd.DataFrame:
    """Integrate multi-omics features into a single matrix."""
    # Transpose expression and proteomics (genes/proteins as columns)
    X = pd.concat([expr.T, prot.T, snp], axis=1, join="inner")
    # Ensure all column names are strings (some may be integers)
    X.columns = X.columns.astype(str)
    # Fill NaN values with column mean (simple imputation for RPPA missing values)
    X = X.fillna(X.mean())
    logging.info(f"Integrated feature matrix: {X.shape}")
    return X


# Test feature selection
expr_top200 = select_top_variable_features(expr_df, 200, "genes")
prot_top100 = select_top_variable_features(prot_df, 100, "proteins")
X_test = integrate_features(expr_top200, prot_top100, snp_df)
print(f"Test integration shape: {X_test.shape}")
print(f"NaN values: {X_test.isna().sum().sum()}")

02:45:46 | INFO | Selected top 200 variable genes
02:45:46 | INFO | Selected top 100 variable proteins
02:45:46 | INFO | Integrated feature matrix: (277, 400)


Test integration shape: (277, 400)
NaN values: 0


## 5. Experiment A: Feature Threshold Comparison

In [5]:
def run_feature_threshold_experiment(
    expr_df: pd.DataFrame,
    prot_df: pd.DataFrame,
    snp_df: pd.DataFrame,
    y: pd.Series,
    gene_thresholds: List[int],
    protein_thresholds: List[int],
    seed: int,
) -> pd.DataFrame:
    """Test different feature selection thresholds."""

    results = []

    for n_genes in gene_thresholds:
        for n_proteins in protein_thresholds:
            # Select features
            expr_top = select_top_variable_features(
                expr_df, min(n_genes, len(expr_df)), "genes"
            )
            prot_top = select_top_variable_features(
                prot_df, min(n_proteins, len(prot_df)), "proteins"
            )

            # Integrate
            X = integrate_features(expr_top, prot_top, snp_df)

            # Scale and PCA
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            pca = PCA(n_components=min(10, X_scaled.shape[1], X_scaled.shape[0] - 1))
            X_pca = pca.fit_transform(X_scaled)

            # Evaluate with Random Forest (5-fold CV)
            rf = RandomForestClassifier(n_estimators=100, random_state=seed)
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            cv_scores = cross_val_score(rf, X_pca, y, cv=cv, scoring="accuracy")

            results.append(
                {
                    "n_genes": n_genes,
                    "n_proteins": n_proteins,
                    "total_features": X.shape[1],
                    "pca_variance_explained": pca.explained_variance_ratio_.sum(),
                    "cv_accuracy_mean": cv_scores.mean(),
                    "cv_accuracy_std": cv_scores.std(),
                }
            )

            print(
                f"Genes={n_genes}, Proteins={n_proteins}: Accuracy={cv_scores.mean():.3f} ± {cv_scores.std():.3f}"
            )

    return pd.DataFrame(results)


print("Running feature threshold experiments...")
threshold_results = run_feature_threshold_experiment(
    expr_df,
    prot_df,
    snp_df,
    y,
    CONFIG.gene_thresholds,
    CONFIG.protein_thresholds,
    CONFIG.random_seed,
)
threshold_results

02:45:46 | INFO | Selected top 100 variable genes
02:45:46 | INFO | Selected top 50 variable proteins
02:45:46 | INFO | Integrated feature matrix: (277, 250)


Running feature threshold experiments...


02:45:47 | INFO | Selected top 100 variable genes
02:45:47 | INFO | Selected top 100 variable proteins
02:45:47 | INFO | Integrated feature matrix: (277, 300)


Genes=100, Proteins=50: Accuracy=0.978 ± 0.013


02:45:47 | INFO | Selected top 100 variable genes
02:45:47 | INFO | Selected top 150 variable proteins
02:45:47 | INFO | Integrated feature matrix: (277, 350)


...

02:45:55 | INFO | Integrated feature matrix: (277, 1050)
02:45:56 | INFO | Selected top 800 variable genes
02:45:56 | INFO | Selected top 200 variable proteins


Genes=800, Proteins=150: Accuracy=0.982 ± 0.000


02:45:56 | INFO | Integrated feature matrix: (277, 1100)


Genes=800, Proteins=200: Accuracy=0.982 ± 0.000


,n_genes,n_proteins,total_features,pca_variance_explained,cv_accuracy_mean,cv_accuracy_std
0,100,50,250,0.348157,0.978312,0.013469
1,100,100,300,0.356998,0.971169,0.024205
2,100,150,350,0.372937,0.967597,0.034722
3,100,200,400,0.387586,0.949610,0.030619
4,200,50,350,0.367976,0.981883,0.011500
5,200,100,400,0.367271,0.974675,0.014450
6,200,150,450,0.375716,0.967597,0.020759
7,200,200,500,0.385907,0.967597,0.020759
8,500,50,650,0.434000,0.978312,0.007339
9,500,100,700,0.421129,0.978312,0.007339


## 6. Experiment B: Clustering Algorithm Comparison

In [6]:
def evaluate_clustering(
    X_pca: np.ndarray, y_true: pd.Series, seed: int
) -> Dict[str, Dict]:
    """Evaluate different clustering algorithms."""

    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

    results = {}

    # K-Means
    kmeans = KMeans(n_clusters=2, random_state=seed, n_init=10)
    kmeans_labels = kmeans.fit_predict(X_pca)
    results["KMeans"] = {
        "labels": kmeans_labels,
        "ari": adjusted_rand_score(y_true, kmeans_labels),
        "nmi": normalized_mutual_info_score(y_true, kmeans_labels),
    }

    # Hierarchical (Agglomerative)
    hier = AgglomerativeClustering(n_clusters=2, linkage="ward")
    hier_labels = hier.fit_predict(X_pca)
    results["Hierarchical"] = {
        "labels": hier_labels,
        "ari": adjusted_rand_score(y_true, hier_labels),
        "nmi": normalized_mutual_info_score(y_true, hier_labels),
    }

    # DBSCAN (with optimized epsilon)
    from sklearn.neighbors import NearestNeighbors

    nn = NearestNeighbors(n_neighbors=5)
    nn.fit(X_pca)
    distances, _ = nn.kneighbors(X_pca)
    eps_estimate = np.median(distances[:, -1])

    dbscan = DBSCAN(eps=eps_estimate, min_samples=5)
    dbscan_labels = dbscan.fit_predict(X_pca)

    # Handle noise points (-1) for metrics
    mask = dbscan_labels >= 0
    if mask.sum() > 0 and len(np.unique(dbscan_labels[mask])) > 1:
        results["DBSCAN"] = {
            "labels": dbscan_labels,
            "ari": adjusted_rand_score(y_true[mask], dbscan_labels[mask]),
            "nmi": normalized_mutual_info_score(y_true[mask], dbscan_labels[mask]),
            "noise_points": (~mask).sum(),
        }
    else:
        results["DBSCAN"] = {
            "labels": dbscan_labels,
            "ari": 0.0,
            "nmi": 0.0,
            "noise_points": (~mask).sum(),
        }

    return results


# Prepare integrated data with best threshold
expr_best = select_top_variable_features(expr_df, 200, "genes")
prot_best = select_top_variable_features(prot_df, 100, "proteins")
X_integrated = integrate_features(expr_best, prot_best, snp_df)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_integrated)

pca = PCA(n_components=CONFIG.n_pca_components)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.3f}")

# Run clustering comparison
clustering_results = evaluate_clustering(X_pca, y, CONFIG.random_seed)

print("\nClustering Results:")
for method, metrics in clustering_results.items():
    print(f"  {method}: ARI={metrics['ari']:.3f}, NMI={metrics['nmi']:.3f}")

02:45:57 | INFO | Selected top 200 variable genes
02:45:57 | INFO | Selected top 100 variable proteins
02:45:57 | INFO | Integrated feature matrix: (277, 400)


PCA variance explained: 0.367

Clustering Results:
  KMeans: ARI=0.904, NMI=0.834
  Hierarchical: ARI=0.903, NMI=0.805
  DBSCAN: ARI=0.925, NMI=0.858


## 7. Experiment C: Machine Learning Model Comparison

In [7]:
def evaluate_ml_models(X: np.ndarray, y: pd.Series, seed: int) -> pd.DataFrame:
    """Compare different ML classification models."""

    models = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=seed),
        "SVM_RBF": SVC(kernel="rbf", random_state=seed),
        "SVM_Linear": SVC(kernel="linear", random_state=seed),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=100, random_state=seed
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32), max_iter=500, random_state=seed
        ),
    }

    results = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for name, model in models.items():
        # Cross-validation scores
        accuracy_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
        precision_scores = cross_val_score(model, X, y, cv=cv, scoring="precision")
        recall_scores = cross_val_score(model, X, y, cv=cv, scoring="recall")
        f1_scores = cross_val_score(model, X, y, cv=cv, scoring="f1")

        results.append(
            {
                "model": name,
                "accuracy": f"{accuracy_scores.mean():.3f} ± {accuracy_scores.std():.3f}",
                "precision": f"{precision_scores.mean():.3f} ± {precision_scores.std():.3f}",
                "recall": f"{recall_scores.mean():.3f} ± {recall_scores.std():.3f}",
                "f1_score": f"{f1_scores.mean():.3f} ± {f1_scores.std():.3f}",
                "accuracy_mean": accuracy_scores.mean(),
                "f1_mean": f1_scores.mean(),
            }
        )

        print(
            f"{name}: Accuracy={accuracy_scores.mean():.3f}, F1={f1_scores.mean():.3f}"
        )

    return pd.DataFrame(results)


print("Running ML model comparison...")
ml_results = evaluate_ml_models(X_pca, y, CONFIG.random_seed)
ml_results[["model", "accuracy", "precision", "recall", "f1_score"]]

Running ML model comparison...
RandomForest: Accuracy=0.975, F1=0.938
SVM_RBF: Accuracy=0.975, F1=0.938
SVM_Linear: Accuracy=0.964, F1=0.909
GradientBoosting: Accuracy=0.964, F1=0.908
MLP: Accuracy=0.975, F1=0.936


,model,accuracy,precision,recall,f1_score
0,RandomForest,0.975 ± 0.014,0.916 ± 0.049,0.964 ± 0.045,0.938 ± 0.036
1,SVM_RBF,0.975 ± 0.018,0.932 ± 0.063,0.945 ± 0.045,0.938 ± 0.045
2,SVM_Linear,0.964 ± 0.028,0.896 ± 0.068,0.927 ± 0.106,0.909 ± 0.075
3,GradientBoosting,0.964 ± 0.016,0.913 ± 0.053,0.909 ± 0.081,0.908 ± 0.042
4,MLP,0.975 ± 0.018,0.928 ± 0.036,0.945 ± 0.073,0.936 ± 0.049


## 8. Validate with Unit Tests

In [8]:
# Validation assertions
assert len(threshold_results) > 0, "Threshold experiment must produce results"
assert all(
    col in threshold_results.columns for col in ["n_genes", "cv_accuracy_mean"]
), "Threshold results must have required columns"

assert len(clustering_results) == 3, "Must test 3 clustering algorithms"
assert all(
    method in clustering_results for method in ["KMeans", "Hierarchical", "DBSCAN"]
), "Must include all specified clustering methods"

assert len(ml_results) == 5, "Must test 5 ML models"
assert ml_results["accuracy_mean"].max() > 0.5, "Best model should beat random baseline"

print("[OK] All validation tests passed!")

[OK] All validation tests passed!


## 9. Export Results

In [9]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export threshold experiment results
threshold_file = EXPORT_DIR / "task2_threshold_experiment.csv"
threshold_results.to_csv(threshold_file, index=False)

# Export clustering results
clustering_df = pd.DataFrame(
    [
        {"method": k, "ari": v["ari"], "nmi": v["nmi"]}
        for k, v in clustering_results.items()
    ]
)
clustering_file = EXPORT_DIR / "task2_clustering_comparison.csv"
clustering_df.to_csv(clustering_file, index=False)

# Export ML model results
ml_file = EXPORT_DIR / "task2_ml_comparison.csv"
ml_results.to_csv(ml_file, index=False)

# Export integrated feature matrix
integrated_file = EXPORT_DIR / "task2_integrated_features.csv"
X_integrated.to_csv(integrated_file)

# Export PCA-transformed data
pca_df = pd.DataFrame(
    X_pca,
    index=X_integrated.index,
    columns=[f"PC{i + 1}" for i in range(X_pca.shape[1])],
)
pca_file = EXPORT_DIR / "task2_pca_features.csv"
pca_df.to_csv(pca_file)

print(f"[OK] Threshold results saved to: {threshold_file.resolve()}")
print(f"[OK] Clustering comparison saved to: {clustering_file.resolve()}")
print(f"[OK] ML comparison saved to: {ml_file.resolve()}")
print(f"[OK] Integrated features saved to: {integrated_file.resolve()}")
print(f"[OK] PCA features saved to: {pca_file.resolve()}")

[OK] Threshold results saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task2_threshold_experiment.csv
[OK] Clustering comparison saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task2_clustering_comparison.csv
[OK] ML comparison saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task2_ml_comparison.csv
[OK] Integrated features saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task2_integrated_features.csv
[OK] PCA features saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task2_pca_features.csv


In [10]:
# Summary statistics
print("\n" + "=" * 60)
print("TASK 2 PARAMETER EXPERIMENT SUMMARY")
print("=" * 60)

best_threshold = threshold_results.loc[threshold_results["cv_accuracy_mean"].idxmax()]
print(f"\nBest Feature Threshold:")
print(
    f"  Genes: {int(best_threshold['n_genes'])}, Proteins: {int(best_threshold['n_proteins'])}"
)
print(
    f"  Accuracy: {best_threshold['cv_accuracy_mean']:.3f} ± {best_threshold['cv_accuracy_std']:.3f}"
)

best_clustering = max(clustering_results.items(), key=lambda x: x[1]["ari"])
print(f"\nBest Clustering Method:")
print(
    f"  {best_clustering[0]}: ARI={best_clustering[1]['ari']:.3f}, NMI={best_clustering[1]['nmi']:.3f}"
)

best_ml = ml_results.loc[ml_results["f1_mean"].idxmax()]
print(f"\nBest ML Model:")
print(f"  {best_ml['model']}: Accuracy={best_ml['accuracy']}, F1={best_ml['f1_score']}")

print("=" * 60)


TASK 2 PARAMETER EXPERIMENT SUMMARY

Best Feature Threshold:
  Genes: 800, Proteins: 50
  Accuracy: 0.986 ± 0.007

Best Clustering Method:
  DBSCAN: ARI=0.925, NMI=0.858

Best ML Model:
  RandomForest: Accuracy=0.975 ± 0.014, F1=0.938 ± 0.036
